In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F


In [2]:
!nvidia-smi

Sat Jan 17 23:46:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!gpustat

/bin/bash: line 1: gpustat: command not found


### I. Loading Dataset - CIFAR-10 & CIFAR-100

In [4]:
import torch 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt 
from PIL import Image
import numpy as np 
import os 
from pathlib import Path 

# Huggingface Datasets + GPT Tokenizer
from datasets import load_dataset, load_from_disk
from transformers import GPT2Tokenizer 

class AddGaussianNoise(object):
    def __init__(self, mean=0., std=0.1):
        self.mean = mean
        self.std = std
        
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean

class CIFAR100(datasets.CIFAR100): 
    def __init__(self, args): 

        self.CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
        self.CIFAR100_STD = (0.2675, 0.2565, 0.2761)

        # Train Transformations
        self.train_transform_list = []
        if args.resize: 
            self.train_transform_list += [
                transforms.Resize((args.resize, args.resize))
                ]
        if args.augment:
            self.train_transform_list += [
                transforms.RandomCrop(args.resize if args.resize else 32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            ]
            
        self.train_transform_list += [
            transforms.ToTensor(),
            transforms.Normalize(mean=self.CIFAR100_MEAN, std=self.CIFAR100_STD),
            ]
        if args.noise > 0.0:
            self.train_transform_list += [AddGaussianNoise(mean=0., std=args.noise)]

        # Test Transformations
        self.test_transform_list = []
        if args.resize: 
            self.test_transform_list += [transforms.Resize((args.resize, args.resize))]
        self.test_transform_list += [
            transforms.ToTensor(),
            transforms.Normalize(mean=self.CIFAR100_MEAN, std=self.CIFAR100_STD)
        ]
        
        # Define transforms
        train_transform = transforms.Compose(self.train_transform_list)
        test_transform = transforms.Compose(self.test_transform_list)

        # Load Datasets
        self.train_data = datasets.CIFAR100(root=args.data_path, train=True, download=True, transform=train_transform)
        self.test_data = datasets.CIFAR100(root=args.data_path, train=False, download=True, transform=test_transform)

        # Data Loaders
        self.train_loader = DataLoader(dataset=self.train_data, batch_size=args.batch_size, shuffle=True, num_workers=6, pin_memory=True)
        self.test_loader = DataLoader(dataset=self.test_data, batch_size=args.batch_size, shuffle=False, num_workers=6, pin_memory=True)

        # Set image size and number of classes
        self.img_size = (3, args.resize, args.resize) if args.resize else (3, 32, 32)
        self.num_classes = 100

    def shape(self):
        return self.train_data[0][0].shape
        
class CIFAR10(datasets.CIFAR10): 
    def __init__(self, args):

        self.CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
        self.CIFAR10_STD = (0.2470, 0.2435, 0.2616)

        # Transformations
        # Train Transformations
        self.train_transform_list = []
        if args.resize: 
            self.train_transform_list += [transforms.Resize((args.resize, args.resize))]
        if args.augment:
            self.train_transform_list += [
                transforms.RandomCrop(args.resize if args.resize else 32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            ]
        self.train_transform_list += [
            transforms.ToTensor(),
            transforms.Normalize(mean=self.CIFAR10_MEAN, std=self.CIFAR10_STD),
            ]
        if args.noise > 0.0:
            self.train_transform_list += [AddGaussianNoise(mean=0., std=args.noise)]

        # Test Transformations
        self.test_transform_list = []
        if args.resize: 
            self.test_transform_list += [transforms.Resize((args.resize, args.resize))]
        self.test_transform_list += [
            transforms.ToTensor(),
            transforms.Normalize(mean=self.CIFAR10_MEAN, std=self.CIFAR10_STD)
        ]
        
        # Define transforms
        train_transform = transforms.Compose(self.train_transform_list)
        test_transform = transforms.Compose(self.test_transform_list)

        # Load Datasets
        self.train_data = datasets.CIFAR10(root=args.data_path, train=True, download=True, transform=train_transform)
        self.test_data = datasets.CIFAR10(root=args.data_path, train=False, download=True, transform=test_transform)

        # Data Loaders
        self.train_loader = DataLoader(dataset=self.train_data, batch_size=args.batch_size, shuffle=True, num_workers=6, pin_memory=True)
        self.test_loader = DataLoader(dataset=self.test_data, batch_size=args.batch_size, shuffle=False, num_workers=6, pin_memory=True)

        # Set image size and number of classes
        self.img_size = (3, args.resize, args.resize) if args.resize else (3, 32, 32)
        self.num_classes = 10

    def shape(self): 
        return self.train_data[0][0].shape
    

### II. Training + Evaluation Loops

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import time 
from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast
import torch.profiler
import random 
from transformers import get_linear_schedule_with_warmup

def print_cuda_info():
    """
    Print information about the available CUDA GPU(s).

    Especially useful for debugging on HPC, Google Colab, and other environments. 
    """
    import torch 

    # Check if CUDA is available
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    # Get number of GPUs
    print(f"Number of GPUs: {torch.cuda.device_count()}")

    # Get current GPU name
    if torch.cuda.is_available():
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")
        
        # Get memory info (in bytes)
        print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        print(f"Allocated memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"Cached memory: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

def write_to_file(file_path, data):
    """
    Write data to a file in a readable format.

    Args:
        file_path (str): The path to the file.
        data: The data to write to the file (can be various types).
    """
    with open(file_path, 'w') as file:
        if isinstance(data, list):
            # For lists like train_eval_results
            for item in data:
                file.write(f"{item}\n")
        elif hasattr(data, '__dict__'):
            # For objects like args
            for key, value in vars(data).items():
                file.write(f"{key}: {value}\n")
        elif isinstance(data, nn.Module):
            # For PyTorch models
            file.write(str(data))
        else:
            # Default case
            file.write(str(data))
            file.write("\n")

def set_seed(seed):
    """
    Set the random seed for reproducibility.

    Args:
        seed (int): The seed value.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
def Train_Eval(args, 
               model: nn.Module, 
               train_loader, 
               test_loader
               ):
    
    if args.seed != 0:
        set_seed(args.seed)

    # Loss Criterion
    if args.criterion == 'CrossEntropy':
        criterion = nn.CrossEntropyLoss()
    elif args.criterion == 'MSE':
        criterion = nn.MSELoss()
            
    # Optimizer 
    if args.optimizer == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    elif args.optimizer == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.weight_decay)
    elif args.optimizer == 'adamw':
        optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        

    # Learning Rate Scheduler
    scheduler = None
    if args.scheduler == 'step':
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=args.lr_step, gamma=args.lr_gamma)
    elif args.scheduler == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.num_epochs)
    elif args.scheduler == 'plateau':
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=5)
        
    # Device
    device = args.device
    model.to(device)
    criterion.to(device)

    scaler = GradScaler() if args.use_amp else None 
        
    epoch_results = []

    ## [GFLOPS] Computation using PyTorch Profiler ##
    try:
        input_tensor, _ = next(iter(train_loader))
        input_tensor = input_tensor.to(device)

        # Profile a single forward pass
        with torch.profiler.profile(
            activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
            with_flops=True
        ) as prof:
            with torch.no_grad():
                model(input_tensor[0:1])

        total_flops = sum(event.flops for event in prof.key_averages())
        if total_flops > 0:
            gflops = total_flops / 1e9
            params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
            print(f"Model Complexity (Profiler):")
            print(f"   - Total Parameters: {params:.8f} M")
            print(f"   - GFLOPs: {gflops:.8f}")
            epoch_results.append(f"Model Complexity (Profiler): GFLOPs: {gflops:.8f}, Trainable Parameters: {params:.8f} M")
    except Exception as e:
        print(f"Could not calculate GFLOPs with PyTorch Profiler: {e}")
    
    # Training Loop
    epoch_times = [] # Average Epoch Time 
    max_accuracy = 0.0 
    max_epoch = 0
    
    for epoch in range(args.num_epochs):
        # Model Training
        model.train() 
        train_running_loss = 0.0
        test_running_loss = 0.0
        epoch_result = ""
        
        start_time = time.time()

        train_top1_5 = [0, 0]
        for images, labels in train_loader: 
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # use mixed precision training
            if args.use_amp:
                with autocast(device_type=args.device):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                if args.clip_grad_norm:
                    scaler.unscale_(optimizer) # Unscale gradients before clipping
                    torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:    
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                if args.clip_grad_norm:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_grad_norm)
                optimizer.step()            

            top1, top5 = accuracy(outputs, labels, topk=(1, 5))
            train_top1_5[0] += top1.item()
            train_top1_5[1] += top5.item()
            train_running_loss += loss.item()

        train_top1_5[0] /= len(train_loader)
        train_top1_5[1] /= len(train_loader)
        
        # Model Evaluation 
        model.eval()
        test_top1_5 = [0, 0]
        with torch.no_grad():
            for images, labels in test_loader: 
                images, labels = images.to(device), labels.to(device)
                if args.use_amp:
                    with autocast(device_type=args.device):
                        outputs = model(images)
                else: 
                    outputs = model(images)
                loss = criterion(outputs, labels)
                test_running_loss += loss.item()

                top1, top5 = accuracy(outputs, labels, topk=(1, 5))
                test_top1_5[0] += top1.item()
                test_top1_5[1] += top5.item()
        
        test_top1_5[0] /= len(test_loader)
        test_top1_5[1] /= len(test_loader)

        # Single Epoch Duration
        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)

        # Save Epoch Results
        epoch_results.append(f"[Epoch {epoch+1:03d}] Time: {epoch_time:.4f}s | [Train] Loss: {train_running_loss/len(train_loader):.8f} Accuracy: Top1: {train_top1_5[0]:.4f}%, Top5: {train_top1_5[1]:.4f}% | [Test] Loss: {test_running_loss/len(test_loader):.8f} Accuracy: Top1: {test_top1_5[0]:.4f}%, Top5: {test_top1_5[1]:.4f}%")
        print(epoch_results[-1])
        
        # Max Accuracy Check
        if test_top1_5[0] > max_accuracy:
            max_accuracy = test_top1_5[0]
            max_epoch = epoch + 1    
            
        # Learning Rate Scheduler Step
        if scheduler: 
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(test_top1_5[0])
            else:
                scheduler.step()
                
                
    epoch_results.append(f"\nAverage Epoch Time: {sum(epoch_times) / len(epoch_times):.4f}s")
    epoch_results.append(f"Max Accuracy: {max_accuracy:.4f}% at Epoch {max_epoch}")
    
    return epoch_results

def accuracy(output, target, topk=(1,)):
    """Computes the top-1 and top-5 accuracy of the model."""
    maxk = min(max(topk), output.size()[1])
    batch_size = target.size(0)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.reshape(1, -1).expand_as(pred))
    return [correct[:min(k, maxk)].reshape(-1).float().sum(0) * 100. / batch_size for k in topk] # [72.5, 91.3] - [top1, top5]

### III. Model Profiling

In [6]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
from torchsummary import summary
import numpy as np 

"""
Fully Convolutional Vision Transformer (FCVT) Model Class
"""

class PatchEmbedding2D(nn.Module):
    def __init__(self, d_hidden, img_size, patch_size, n_channels=3):
        super(PatchEmbedding2D, self).__init__()

        self.d_hidden = d_hidden # Dimensionality of Model 
        self.img_size = img_size # Size of Image
        self.patch_size = patch_size # Patch Size 
        self.n_channels = n_channels # Number of Channels in Image
        
        self.linear_projection = nn.Conv2d(in_channels=n_channels, out_channels=d_hidden, kernel_size=patch_size, stride=patch_size) # Linear Projection Layer
        self.norm = nn.LayerNorm(d_hidden) # Normalization Layer

    def forward(self, x):
        x = self.linear_projection(x) # (B, C, H, W) -> (B, d_hidden, H', W')
        x = x.permute(0, 2, 3, 1) # (B, d_hidden, H', W') -> (B, H', W', d_hidden)
        x = self.norm(x) # (B, H', W', d_hidden) -> (B, H', W', d_hidden)
        x = x.permute(0, 3, 1, 2) # (B, H', W', d_hidden) -> (B, d_hidden, H', W')
        return x

class PositionalEncoding2D(nn.Module):
    def __init__(self, d_hidden, height, width):
        super(PositionalEncoding2D, self).__init__() 

        self.d_hidden = d_hidden 
        self.height = height 
        self.width = width 

        # Class token 
        self.class_token = nn.Parameter(torch.randn(1, d_hidden, 1, 1))

        # Learnable 2D positional embeddings for width + 1 (patches + class token)
        self.positional_encoding = nn.Parameter(torch.randn(1, d_hidden, height, width + 1))

    def forward(self, x):
        B, _, H, _ = x.shape 

        class_tokens = self.class_token.expand(B, -1, H, -1) 

        # Concatenate along width dimension 
        x = torch.cat([class_tokens, x], dim=3) # (B, d_hidden, H, W + 1)

        # Add positional encoding 
        x = x + self.positional_encoding # (B, d_hidden, H, W + 1)
        return x

# TODO is not working, need to fix later
class SinusoidalPositionalEncoding2D(nn.Module):
    def __init__(self, d_hidden, height, width, temperature = 10000):
        super(SinusoidalPositionalEncoding2D, self).__init__() 

        self.d_hidden = d_hidden 
        self.height = height 
        self.width = width 
        self.temperature = temperature

        # Class token 
        self.class_token = nn.Parameter(torch.randn(1, d_hidden, 1, 1))

        # 2D sinusoidal positional encoding 
        # split d_hidden: half for height encoding, half for width encoding 
        pe = torch.zeros(d_hidden, height, width + 1) # +1 for class token column

        d_model_half = d_hidden // 2
        
        # Compute division term
        div_term = torch.exp(torch.arange(0, d_model_half, 2).float() * 
                           (-np.log(self.temperature) / d_model_half))
        
        # Height encoding (first half of channels)
        pos_h = torch.arange(0, height).unsqueeze(1).float()  # (H, 1)
        pe[:d_model_half:2, :, :] = torch.sin(pos_h * div_term).unsqueeze(-1).repeat(1, 1, width + 1)
        pe[1:d_model_half:2, :, :] = torch.cos(pos_h * div_term).unsqueeze(-1).repeat(1, 1, width + 1)
        
        # Width encoding (second half of channels)
        # Note: width + 1 to include class token position (position 0)
        pos_w = torch.arange(0, width + 1).unsqueeze(1).float()  # (W+1, 1)
        pe[d_model_half::2, :, :] = torch.sin(pos_w * div_term).unsqueeze(1).repeat(1, height, 1)
        pe[d_model_half+1::2, :, :] = torch.cos(pos_w * div_term).unsqueeze(1).repeat(1, height, 1)
        
        # Register as buffer (non-trainable)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, d_hidden, H, W+1)

    def forward(self, x):
        # x shape: (B, d_hidden, H, W)
        B, _, H, _ = x.shape 

        # Expand class token for batch and height
        class_tokens = self.class_token.expand(B, -1, H, -1)  # (B, d_hidden, H, 1)

        # Concatenate along width dimension 
        x = torch.cat([class_tokens, x], dim=3)  # (B, d_hidden, H, W + 1)

        # Add sinusoidal positional encoding (non-trainable)
        x = x + self.pe
        
        return x

class ConvolutionalAttention2D_Old(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout):
        super(ConvolutionalAttention2D_Old, self).__init__()

        self.d_hidden = d_hidden
        self.dropout = nn.Dropout(attention_dropout)

        # Pointwise Convolution to keep dimension
        self.W_q = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_k = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_v = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_o = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1)

        # Pointwise Convolution for KQV
        self.W_kqv = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_kqv.weight.requires_grad = False

    def phi(self, x):
        return F.elu(x) + 1
    
    def forward(self, x, mask = None):
        B, _, _, _ = x.shape 
        
        q = self.W_q(x) # (B, d_hidden, H, W)
        k = self.W_k(x) # (B, d_hidden, H, W)
        v = self.W_v(x) # (B, d_hidden, H, W)

        # Need Phi_k and Phi_q 
        ## USING PLACEHOLDERS FOR NOW
        phi_q = self.phi(q) 
        phi_k = self.phi(k)
        phi_v = self.phi(v) 

        qv_matrix = torch.einsum('bchw,bdhw->bcd', phi_q, phi_v)  # (32, n_channel_q, n_channel_v)

        # for each batch index, insert for weight and convolve on phi_k
        attended_batch = []
        for i in range(B):
            self.W_kqv.weight.data = qv_matrix[i].unsqueeze(-1).unsqueeze(-1) 
            k_batch_index = phi_k[i].unsqueeze(0)
            out = self.W_kqv(k_batch_index)
            attended_batch.append(out) 

        combined_batch = torch.cat(attended_batch, dim=0)
        out = self.W_o(combined_batch) 
        out = self.dropout(out)
        return out


class ConvolutionalAttention2D(nn.Module):
    """
    Docstring for ConvolutionalAttention2D
    - Dynamic Conv2d Kernel Linear Attention from original idea
    """
    
    def __init__(self, d_hidden, num_heads, attention_dropout):
        super(ConvolutionalAttention2D, self).__init__()

        self.d_hidden = d_hidden
        self.dropout = nn.Dropout(attention_dropout)

        # Pointwise Convolution to keep dimension
        self.W_q = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_k = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_v = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_o = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1)

    def phi(self, x):
        return F.elu(x) + 1
    
    def forward(self, x, mask = None):
        B, C, H, W = x.shape
        
        q = self.W_q(x) # (B, d_hidden, H, W)
        k = self.W_k(x) # (B, d_hidden, H, W)
        v = self.W_v(x) # (B, d_hidden, H, W)

        phi_q = self.phi(q) 
        phi_k = self.phi(k)
        phi_v = self.phi(v) 

        # Comput Q^T @ V for each batch -> (B, C, C)
        qv_matrix = torch.einsum('bchw,bdhw->bcd', phi_q, phi_v)  # (B, n_channel_q, n_channel_v)

        # Reshape for grouped convolution 
        weight = qv_matrix.reshape(B * C, C, 1, 1)  # (B * n_channel_q, n_channel_v, 1, 1)
        k_grouped = phi_k.reshape(1, B * C, H, W)  # (1, B * n_channel_k, H, W )

        out = F.conv2d(k_grouped, weight, groups=B)  # (1, B * n_channel_q, H, W)
        out = out.reshape(B, C, H, W)  # (B, n_channel_q, H, W)

        out = self.W_o(out) 
        return self.dropout(out) 

class FCVTEncoder(nn.Module):
    def __init__(self, d_hidden, d_mlp, n_heads, dropout, attention_dropout):
        super(FCVTEncoder, self).__init__()
        
        self.d_hidden = d_hidden 
        self.d_mlp = d_mlp 
        self.n_heads = n_heads 
        self.dropout = dropout 
        self.attention_dropout = attention_dropout 

        self.conv_attention = ConvolutionalAttention2D(d_hidden, n_heads, attention_dropout)

        # Norm and Dropout 
        self.norm1 = nn.LayerNorm(d_hidden)
        self.norm2 = nn.LayerNorm(d_hidden)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout) 

        # MLP 
        self.mlp = nn.Sequential(
            nn.Conv2d(d_hidden, d_mlp, kernel_size=1, stride=1), 
            nn.GELU(), 
            nn.Dropout(dropout), 
            nn.Conv2d(d_mlp, d_hidden, kernel_size=1, stride=1)
        )

    def forward(self, x):
        x_norm = x.permute(0, 2, 3, 1) # (B, d_hidden, H', W') -> (B, H', W', d_hidden)
        x_norm = self.norm1(x_norm) # (B, H', W', d_hidden) -> (B, H', W', d_hidden)
        x_norm = x_norm.permute(0, 3, 1, 2) # (B, H', W', d_hidden) -> (B, d_hidden, H', W')

        attn_output = self.conv_attention(x_norm)
        x = x + self.dropout1(attn_output)

        # Post-Norm Feed Forward Network
        x_norm = x.permute(0, 2, 3, 1) # (B, d_hidden, H', W') -> (B, H', W', d_hidden)
        x_norm = self.norm2(x_norm) # (B, H', W', d_hidden) -> (B, H', W', d_hidden)
        x_norm = x_norm.permute(0, 3, 1, 2) # (B, H', W', d_hidden) -> (B, d_hidden, H', W')
        mlp_output = self.mlp(x_norm)
        x = x + self.dropout2(mlp_output)
        return x 

class FCVT(nn.Module):
    def __init__(self, d_hidden, d_mlp, img_size, n_classes, n_heads, patch_size, n_channels, n_layers, dropout, attention_dropout):
        super(FCVT, self).__init__() 
        assert img_size[1] % patch_size == 0 and img_size[2] % patch_size == 0, "img_size dimensions must be divisible by patch_size dimensions"
        assert d_hidden % n_heads == 0, "d_hidden must be divisible by n_heads"

        self.model = "FCVT"
        self.d_hidden = d_hidden 
        self.d_mlp = d_mlp
        self.img_size = img_size[1:]
        self.n_classes = n_classes 
        self.n_heads = n_heads
        self.patch_size = (patch_size, patch_size)
        self.n_channels = n_channels
        self.n_layers = n_layers
        self.dropout = dropout 
        self.attention_dropout = attention_dropout

        self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1])
        self.max_seq_length = self.n_patches + 1 # + 1 for class token 

        # Layers
        self.patch_embedding = PatchEmbedding2D(d_hidden, img_size, patch_size, n_channels)
        self.positional_encoding = PositionalEncoding2D(d_hidden, img_size[1] // patch_size, img_size[2] // patch_size)
        # self.positional_encoding = SinusoidalPositionalEncoding2D(d_hidden, img_size[1] // patch_size, img_size[2] // patch_size)

        self.transformer_encoder = nn.Sequential(*[FCVTEncoder(
            d_hidden, d_mlp, n_heads, dropout, attention_dropout
        ) for _ in range(self.n_layers)])


        # (Batch, d_hidden, height, width) as input for classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), 
            nn.Flatten(), 
            nn.Linear(d_hidden, n_classes)
        )
        
    def forward(self, x):
        x = self.patch_embedding(x) 
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x) 

        class_output = x[:, :, :, 0:1]
        x = self.classifier(class_output)
        return x
    
    def summary(self): 
        original_device = next(self.parameters()).device
        try:
            self.to("cpu")
            print(f"--- Summary for {self.name} ---")
            summary(self, input_size=self.img_size, device="cpu") 
        except Exception as e:
            print(f"Could not generate summary: {e}")
        finally:
            self.to(original_device)
        
    def parameter_count(self): 
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total_params, trainable_params

### IV. Training Sampling

In [7]:
from types import SimpleNamespace

args = SimpleNamespace(
    # Model Args
    d_hidden=192,
    d_mlp=768,
    img_size=(3, 224, 224),
    n_classes=100,
    n_heads=3,
    patch_size=16,
    n_channels=3,
    n_layers=12,
    dropout=0.1,
    attention_dropout=0.1,
    
    # Training Args
    batch_size=256,
    num_epochs=200,
    lr=1e-3,
    device='cuda',
    
     # Compilation Args
    
    # Dataset Args
    dataset='cifar10',
    data_path='./Data',
    resize=224,
    augment=True,
    noise=0.0,
    
    # Optimizer Args
    optimizer='adamw',
    weight_decay=1e-2,
    
    # Other Args
    use_amp=False,
    clip_grad_norm=1.0,
    criterion='CrossEntropy',
    scheduler='cosine',
    seed=42
)

# 1. First Iteration
- Using ConvolutionalAttention2D

In [ ]:
model = FCVT(
        d_hidden = args.d_hidden, 
        d_mlp=args.d_mlp, 
        img_size=args.img_size,
        n_classes=args.n_classes,
        n_heads=args.n_heads,
        patch_size=args.patch_size,
        n_channels=args.n_channels,
        n_layers=args.n_layers,
        dropout=args.dropout,
        attention_dropout=args.attention_dropout
    )

dataset = CIFAR10(args)


# Train_Eval(args, model, dataset.train_loader, dataset.test_loader)

Model Complexity (Profiler):
   - Total Parameters: 5.53930000 M
   - GFLOPs: 2.66142451
[Epoch 001] Time: 77.7361s | [Train] Loss: 9840.76421334 Accuracy: Top1: 18.3127%, Top5: 65.6804% | [Test] Loss: 1.96879817 Accuracy: Top1: 25.8105%, Top5: 79.4336%
[Epoch 002] Time: 78.7188s | [Train] Loss: 1.84744813 Accuracy: Top1: 29.3977%, Top5: 84.1071% | [Test] Loss: 1.79735122 Accuracy: Top1: 33.1934%, Top5: 85.2637%
[Epoch 003] Time: 77.5457s | [Train] Loss: 1.73010348 Accuracy: Top1: 34.1940%, Top5: 87.0843% | [Test] Loss: 1.67308755 Accuracy: Top1: 35.3027%, Top5: 88.4668%
[Epoch 004] Time: 79.6286s | [Train] Loss: 1.65330890 Accuracy: Top1: 37.6299%, Top5: 88.8277% | [Test] Loss: 1.61154250 Accuracy: Top1: 39.9316%, Top5: 89.8828%
[Epoch 005] Time: 79.2675s | [Train] Loss: 1.57532355 Accuracy: Top1: 41.3297%, Top5: 90.1969% | [Test] Loss: 1.57015085 Accuracy: Top1: 42.5781%, Top5: 90.9375%
[Epoch 006] Time: 78.1198s | [Train] Loss: 1.51067062 Accuracy: Top1: 44.3682%, Top5: 91.4357% | [

: 

# 2. Second Iteration 
- Using ConvolutionalAttention2D_Old instead of ConvolutionalAttention2D

In [ ]:
# PatchEmbedding2D 
import torch 
import torch.nn as nn 
import torch.nn.functional as F
from torchsummary import summary
import numpy as np 

"""
Fully Convolutional Vision Transformer (FCVT) Model Class
"""

class PatchEmbedding2D(nn.Module):
    def __init__(self, d_hidden, img_size, patch_size, n_channels=3):
        super(PatchEmbedding2D, self).__init__()

        self.d_hidden = d_hidden # Dimensionality of Model 
        self.img_size = img_size # Size of Image
        self.patch_size = patch_size # Patch Size 
        self.n_channels = n_channels # Number of Channels in Image
        
        self.linear_projection = nn.Conv2d(in_channels=n_channels, out_channels=d_hidden, kernel_size=patch_size, stride=patch_size) # Linear Projection Layer
        self.norm = nn.LayerNorm(d_hidden) # Normalization Layer

    def forward(self, x):
        x = self.linear_projection(x) # (B, C, H, W) -> (B, d_hidden, H', W')
        x = x.permute(0, 2, 3, 1) # (B, d_hidden, H', W') -> (B, H', W', d_hidden)
        x = self.norm(x) # (B, H', W', d_hidden) -> (B, H', W', d_hidden)
        x = x.permute(0, 3, 1, 2) # (B, H', W', d_hidden) -> (B, d_hidden, H', W')
        return x

class PositionalEncoding2D(nn.Module):
    def __init__(self, d_hidden, height, width):
        super(PositionalEncoding2D, self).__init__() 

        self.d_hidden = d_hidden 
        self.height = height 
        self.width = width 

        # Class token 
        self.class_token = nn.Parameter(torch.randn(1, d_hidden, 1, 1))

        # Learnable 2D positional embeddings for width + 1 (patches + class token)
        self.positional_encoding = nn.Parameter(torch.randn(1, d_hidden, height, width + 1))

    def forward(self, x):
        B, _, H, _ = x.shape 

        class_tokens = self.class_token.expand(B, -1, H, -1) 

        # Concatenate along width dimension 
        x = torch.cat([class_tokens, x], dim=3) # (B, d_hidden, H, W + 1)

        # Add positional encoding 
        x = x + self.positional_encoding # (B, d_hidden, H, W + 1)
        return x

# TODO is not working, need to fix later
class SinusoidalPositionalEncoding2D(nn.Module):
    def __init__(self, d_hidden, height, width, temperature = 10000):
        super(SinusoidalPositionalEncoding2D, self).__init__() 

        self.d_hidden = d_hidden 
        self.height = height 
        self.width = width 
        self.temperature = temperature

        # Class token 
        self.class_token = nn.Parameter(torch.randn(1, d_hidden, 1, 1))

        # 2D sinusoidal positional encoding 
        # split d_hidden: half for height encoding, half for width encoding 
        pe = torch.zeros(d_hidden, height, width + 1) # +1 for class token column

        d_model_half = d_hidden // 2
        
        # Compute division term
        div_term = torch.exp(torch.arange(0, d_model_half, 2).float() * 
                           (-np.log(self.temperature) / d_model_half))
        
        # Height encoding (first half of channels)
        pos_h = torch.arange(0, height).unsqueeze(1).float()  # (H, 1)
        pe[:d_model_half:2, :, :] = torch.sin(pos_h * div_term).unsqueeze(-1).repeat(1, 1, width + 1)
        pe[1:d_model_half:2, :, :] = torch.cos(pos_h * div_term).unsqueeze(-1).repeat(1, 1, width + 1)
        
        # Width encoding (second half of channels)
        # Note: width + 1 to include class token position (position 0)
        pos_w = torch.arange(0, width + 1).unsqueeze(1).float()  # (W+1, 1)
        pe[d_model_half::2, :, :] = torch.sin(pos_w * div_term).unsqueeze(1).repeat(1, height, 1)
        pe[d_model_half+1::2, :, :] = torch.cos(pos_w * div_term).unsqueeze(1).repeat(1, height, 1)
        
        # Register as buffer (non-trainable)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, d_hidden, H, W+1)

    def forward(self, x):
        # x shape: (B, d_hidden, H, W)
        B, _, H, _ = x.shape 

        # Expand class token for batch and height
        class_tokens = self.class_token.expand(B, -1, H, -1)  # (B, d_hidden, H, 1)

        # Concatenate along width dimension 
        x = torch.cat([class_tokens, x], dim=3)  # (B, d_hidden, H, W + 1)

        # Add sinusoidal positional encoding (non-trainable)
        x = x + self.pe
        
        return x

class ConvolutionalAttention2D_Old(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout):
        super(ConvolutionalAttention2D_Old, self).__init__()

        self.d_hidden = d_hidden
        self.dropout = nn.Dropout(attention_dropout)

        # Pointwise Convolution to keep dimension
        self.W_q = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_k = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_v = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_o = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1)

        # Pointwise Convolution for KQV
        self.W_kqv = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_kqv.weight.requires_grad = False

    def phi(self, x):
        return F.elu(x) + 1
    
    def forward(self, x, mask = None):
        B, _, _, _ = x.shape 
        
        q = self.W_q(x) # (B, d_hidden, H, W)
        k = self.W_k(x) # (B, d_hidden, H, W)
        v = self.W_v(x) # (B, d_hidden, H, W)

        # Need Phi_k and Phi_q 
        ## USING PLACEHOLDERS FOR NOW
        phi_q = self.phi(q) 
        phi_k = self.phi(k)
        phi_v = self.phi(v) 

        qv_matrix = torch.einsum('bchw,bdhw->bcd', phi_q, phi_v)  # (32, n_channel_q, n_channel_v)

        # for each batch index, insert for weight and convolve on phi_k
        attended_batch = []
        for i in range(B):
            self.W_kqv.weight.data = qv_matrix[i].unsqueeze(-1).unsqueeze(-1) 
            k_batch_index = phi_k[i].unsqueeze(0)
            out = self.W_kqv(k_batch_index)
            attended_batch.append(out) 

        combined_batch = torch.cat(attended_batch, dim=0)
        out = self.W_o(combined_batch) 
        out = self.dropout(out)
        return out


class ConvolutionalAttention2D(nn.Module):
    """
    Docstring for ConvolutionalAttention2D
    - Dynamic Conv2d Kernel Linear Attention from original idea
    """
    
    def __init__(self, d_hidden, num_heads, attention_dropout):
        super(ConvolutionalAttention2D, self).__init__()

        self.d_hidden = d_hidden
        self.dropout = nn.Dropout(attention_dropout)

        # Pointwise Convolution to keep dimension
        self.W_q = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_k = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_v = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1, bias = False)
        self.W_o = nn.Conv2d(d_hidden, d_hidden, kernel_size = 1, stride = 1)

    def phi(self, x):
        return F.elu(x) + 1
    
    def forward(self, x, mask = None):
        B, C, H, W = x.shape
        
        q = self.W_q(x) # (B, d_hidden, H, W)
        k = self.W_k(x) # (B, d_hidden, H, W)
        v = self.W_v(x) # (B, d_hidden, H, W)

        phi_q = self.phi(q) 
        phi_k = self.phi(k)
        phi_v = self.phi(v) 

        # Comput Q^T @ V for each batch -> (B, C, C)
        qv_matrix = torch.einsum('bchw,bdhw->bcd', phi_q, phi_v)  # (B, n_channel_q, n_channel_v)

        # Reshape for grouped convolution 
        weight = qv_matrix.reshape(B * C, C, 1, 1)  # (B * n_channel_q, n_channel_v, 1, 1)
        k_grouped = phi_k.reshape(1, B * C, H, W)  # (1, B * n_channel_k, H, W )

        out = F.conv2d(k_grouped, weight, groups=B)  # (1, B * n_channel_q, H, W)
        out = out.reshape(B, C, H, W)  # (B, n_channel_q, H, W)

        out = self.W_o(out) 
        return self.dropout(out) 

class FCVTEncoder(nn.Module):
    def __init__(self, d_hidden, d_mlp, n_heads, dropout, attention_dropout):
        super(FCVTEncoder, self).__init__()
        
        self.d_hidden = d_hidden 
        self.d_mlp = d_mlp 
        self.n_heads = n_heads 
        self.dropout = dropout 
        self.attention_dropout = attention_dropout 

        self.conv_attention = ConvolutionalAttention2D_Old(d_hidden, n_heads, attention_dropout)

        # Norm and Dropout 
        self.norm1 = nn.LayerNorm(d_hidden)
        self.norm2 = nn.LayerNorm(d_hidden)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout) 

        # MLP 
        self.mlp = nn.Sequential(
            nn.Conv2d(d_hidden, d_mlp, kernel_size=1, stride=1), 
            nn.GELU(), 
            nn.Dropout(dropout), 
            nn.Conv2d(d_mlp, d_hidden, kernel_size=1, stride=1)
        )

    def forward(self, x):
        x_norm = x.permute(0, 2, 3, 1) # (B, d_hidden, H', W') -> (B, H', W', d_hidden)
        x_norm = self.norm1(x_norm) # (B, H', W', d_hidden) -> (B, H', W', d_hidden)
        x_norm = x_norm.permute(0, 3, 1, 2) # (B, H', W', d_hidden) -> (B, d_hidden, H', W')

        attn_output = self.conv_attention(x_norm)
        x = x + self.dropout1(attn_output)

        # Post-Norm Feed Forward Network
        x_norm = x.permute(0, 2, 3, 1) # (B, d_hidden, H', W') -> (B, H', W', d_hidden)
        x_norm = self.norm2(x_norm) # (B, H', W', d_hidden) -> (B, H', W', d_hidden)
        x_norm = x_norm.permute(0, 3, 1, 2) # (B, H', W', d_hidden) -> (B, d_hidden, H', W')
        mlp_output = self.mlp(x_norm)
        x = x + self.dropout2(mlp_output)
        return x 

class FCVT(nn.Module):
    def __init__(self, d_hidden, d_mlp, img_size, n_classes, n_heads, patch_size, n_channels, n_layers, dropout, attention_dropout):
        super(FCVT, self).__init__() 
        assert img_size[1] % patch_size == 0 and img_size[2] % patch_size == 0, "img_size dimensions must be divisible by patch_size dimensions"
        assert d_hidden % n_heads == 0, "d_hidden must be divisible by n_heads"

        self.model = "FCVT"
        self.d_hidden = d_hidden 
        self.d_mlp = d_mlp
        self.img_size = img_size[1:]
        self.n_classes = n_classes 
        self.n_heads = n_heads
        self.patch_size = (patch_size, patch_size)
        self.n_channels = n_channels
        self.n_layers = n_layers
        self.dropout = dropout 
        self.attention_dropout = attention_dropout

        self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1])
        self.max_seq_length = self.n_patches + 1 # + 1 for class token 

        # Layers
        self.patch_embedding = PatchEmbedding2D(d_hidden, img_size, patch_size, n_channels)
        self.positional_encoding = PositionalEncoding2D(d_hidden, img_size[1] // patch_size, img_size[2] // patch_size)
        # self.positional_encoding = SinusoidalPositionalEncoding2D(d_hidden, img_size[1] // patch_size, img_size[2] // patch_size)

        self.transformer_encoder = nn.Sequential(*[FCVTEncoder(
            d_hidden, d_mlp, n_heads, dropout, attention_dropout
        ) for _ in range(self.n_layers)])


        # (Batch, d_hidden, height, width) as input for classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), 
            nn.Conv2d(d_hidden, n_classes, kernel_size=1, stride=1), 
            nn.Flatten()
        )

        self.name = f"ViT_{self.n_layers}L_{self.n_heads}H_{self.d_hidden}D"
        
    def forward(self, x):
        x = self.patch_embedding(x) 
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x) 

        class_output = x[:, :, :, 0:1]
        x = self.classifier(class_output)
        return x
    
    def summary(self): 
        original_device = next(self.parameters()).device
        try:
            self.to("cpu")
            print(f"--- Summary for {self.name} ---")
            summary(self, input_size=self.img_size, device="cpu") 
        except Exception as e:
            print(f"Could not generate summary: {e}")
        finally:
            self.to(original_device)
        
    def parameter_count(self): 
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total_params, trainable_params

In [ ]:
model = FCVT(
        d_hidden = args.d_hidden, 
        d_mlp=args.d_mlp, 
        img_size=args.img_size,
        n_classes=args.n_classes,
        n_heads=args.n_heads,
        patch_size=args.patch_size,
        n_channels=args.n_channels,
        n_layers=args.n_layers,
        dropout=args.dropout,
        attention_dropout=args.attention_dropout
    )

dataset = CIFAR10(args)


Train_Eval(args, model, dataset.train_loader, dataset.test_loader)

100%|██████████| 170M/170M [00:13<00:00, 12.5MB/s] 


Model Complexity (Profiler):
   - Total Parameters: 5.53930000 M
   - GFLOPs: 2.66142451
[Epoch 001] Time: 172.5602s | [Train] Loss: 16492.35502504 Accuracy: Top1: 13.6452%, Top5: 56.3202% | [Test] Loss: 2.09680499 Accuracy: Top1: 21.8652%, Top5: 72.9102%
[Epoch 002] Time: 171.9428s | [Train] Loss: 2.23939419 Accuracy: Top1: 15.6900%, Top5: 63.6775% | [Test] Loss: 2.97261409 Accuracy: Top1: 10.0977%, Top5: 50.1758%
[Epoch 003] Time: 172.3967s | [Train] Loss: 2.31252214 Accuracy: Top1: 12.9257%, Top5: 57.4753% | [Test] Loss: 2.28899937 Accuracy: Top1: 11.3477%, Top5: 51.6504%
[Epoch 004] Time: 171.9497s | [Train] Loss: 2.35814613 Accuracy: Top1: 13.2390%, Top5: 57.4075% | [Test] Loss: 2.29520327 Accuracy: Top1: 15.1758%, Top5: 55.5078%
[Epoch 005] Time: 172.3099s | [Train] Loss: 2.29662445 Accuracy: Top1: 11.6920%, Top5: 53.6922% | [Test] Loss: 2.29691719 Accuracy: Top1: 13.7598%, Top5: 50.1367%
[Epoch 006] Time: 172.7285s | [Train] Loss: 9.70733291 Accuracy: Top1: 11.3317%, Top5: 53.07

: 

# 3. Third Iteration